In [29]:
# from llama_index.embeddings.cohere import CohereEmbedding
# from llama_index.llms.cohere import Cohere
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json

%matplotlib inline

In [16]:
_ = load_dotenv(".env")

In [17]:
df_store_zara = pd.read_csv('../data/store_zara_data/store_zara.csv', sep=',')
df_store_zara.head()

,brand,url,sku,name,description,price,currency,images,scraped_at,terms,section,error,image_downloads
0,Zara,https://www.zara.com/us/en/basic-puffer-jacket...,272145190-250-2,BASIC PUFFER JACKET,Puffer jacket made of tear-resistant ripstop f...,19.99,USD,['https://static.zara.net/photos///2023/I/0/2/...,2024-02-19T08:50:05.654618,jackets,MAN,NaN,"['e8e4ae57-8b72-44ff-aa5d-84de3ed37d9e', '0cb3..."
1,Zara,https://www.zara.com/us/en/tuxedo-jacket-p0889...,324052738-800-46,TUXEDO JACKET,Straight fit blazer. Pointed lapel collar and ...,169.00,USD,['https://static.zara.net/photos///2024/V/0/1/...,2024-02-19T08:50:06.590930,jackets,MAN,NaN,"['b42b0725-cfe2-4748-af3e-3abe590d83cd', 'ed9c..."
2,Zara,https://www.zara.com/us/en/slim-fit-suit-jacke...,335342680-800-44,SLIM FIT SUIT JACKET,Slim fit jacket. Notched lapel collar. Long sl...,129.00,USD,['https://static.zara.net/photos///2023/I/0/2/...,2024-02-19T08:50:07.301419,jackets,MAN,NaN,"['c27bdddf-2f9c-4693-976f-0d1272212c12', '07e6..."
3,Zara,https://www.zara.com/us/en/stretch-suit-jacket...,328303236-420-44,STRETCH SUIT JACKET,Slim fit jacket made of viscose blend fabric. ...,129.00,USD,['https://static.zara.net/photos///2024/V/0/1/...,2024-02-19T08:50:07.882922,jackets,MAN,NaN,"['0cbc84c8-f9ff-4f02-940c-ff642b5af379', 'dd47..."
4,Zara,https://www.zara.com/us/en/double-faced-jacket...,312368260-800-2,DOUBLE FACED JACKET,Jacket made of faux leather faux shearling wit...,139.00,USD,['https://static.zara.net/photos///2024/V/0/2/...,2024-02-19T08:50:08.453847,jackets,MAN,NaN,"['1ccc9c87-49c9-4825-9d67-4c1f61e5db1f', '1349..."


In [18]:
df_examples = df_store_zara.query('terms == "shoes"').reset_index(drop=True)
df_examples.head(2)

,brand,url,sku,name,description,price,currency,images,scraped_at,terms,section,error,image_downloads
0,Zara,https://www.zara.com/us/en/chunky-sole-chelsea...,311282828-800-39,CHUNKY SOLE CHELSEA BOOTS,Chelsea boots. Shaft with elastic goring on bo...,59.9,USD,['https://static.zara.net/photos///2024/V/1/2/...,2024-02-19T08:59:13.418886,shoes,MAN,NaN,"['8569e73f-b1be-476a-89b4-9afa3f6ff89b', 'd54a..."
1,Zara,https://www.zara.com/us/en/chunky-sole-sneaker...,311282673-001-39,CHUNKY SOLE SNEAKERS,Sneakers. Upper in a combination of materials ...,49.9,USD,['https://static.zara.net/photos///2024/V/1/1/...,2024-02-19T08:59:14.270281,shoes,MAN,NaN,"['38da5001-d5b0-462c-8372-f5381e17ea17', 'af82..."


In [23]:
df_examples.loc[1, 'description']


'Sneakers. Upper in a combination of materials and colors. Lacing with seven pairs of eyelets. Back contrasting color piece. Thick rubber soles with irregular design.'

### Modelo LLM : llama3.2 (3B)

In [66]:
# Se hace uso del modelo LLM llama3.2(3B) de texto para la extraccion de metadatos de la descripción.
# Es un modelo más rápido que los modelos de vision: llama3.2-vision:11b | llava:13b 
llm = Ollama(model="llama3.2")

In [67]:
response = llm.complete("What is the capital of France?")
print(response)

The capital of France is Paris.


In [68]:
Settings.llm = llm

In [69]:
class ShoesDetails(BaseModel):
    """Data model for shoes details."""

    type_shoes: str = Field(
        ..., description="The type of shoes."
    )
    color: str = Field(
        ..., description="The color of the shoes."
    )
    material_sole: str = Field(
        ..., description="The material of the sole."
    )

    limited_edition: str = Field(
        ..., description="If the shoes are limited edition."
    )

## Extract info with pydantic
fuente: https://docs.llamaindex.ai/en/stable/examples/output_parsing/nvidia_output_parsing/

In [ ]:
prompt_template_str = """\
Extract the following information from the description of the shoes.
If the information is not present in the description, return the value 'None'.
Description: {description}
"""
program = LLMTextCompletionProgram.from_defaults(
    output_cls=ShoesDetails,
    prompt_template_str=prompt_template_str,
    verbose=True,
)

In [74]:
### Caso 1: Description
description_example_1 = df_store_zara.query('(terms == "shoes") & (section == "MAN")').reset_index(drop=True).loc[12, 'description']

In [75]:

output = program(descrption=description_example_1)
output

ShoesDetails(type_shoes='Derby', color='None', material_sole='Leather', limited_edition='Limited edition')

In [ ]:
print(output.type_shoes)
print(output.color)
print(output.material_sole)
print(output.limited_edition)

Derby
None
Leather
Limited edition


In [79]:
### Caso 2: Description
description_example_2 = df_store_zara.query('(terms == "shoes") & (section == "MAN")').reset_index(drop=True).loc[11, 'description']

In [82]:
description_example_2

'Running shoes. Made with leather and a combination of materials on the upper. Superimposed pieces at upper. Lacing with six pairs of eyelets. Slightly chunky irregular design soles.\n\nAvailable in different colors. \n\nLimited edition.'

In [80]:
output_2 = program(descrption=description_example_2)
output_2

ShoesDetails(type_shoes='Running shoes', color='Available in different colors', material_sole='Leather and a combination of materials on the upper', limited_edition='Limited edition.')

In [81]:
print(output_2.type_shoes)
print(output_2.color)
print(output_2.material_sole)
print(output_2.limited_edition)

Running shoes
Available in different colors
Leather and a combination of materials on the upper
Limited edition.


In [ ]:
### Caso 3: Description
description_example_3 = df_store_zara.query('(terms == "shoes") & (section == "MAN")').reset_index(drop=True).loc[80, 'description']

In [87]:
description_example_3

'Slip on casual sneakers. Decorative embossed detail at upper. Side elastic goring for ease.'

In [89]:
output_3 = program(descrption=description_example_3)
output_3

ShoesDetails(type_shoes='Slip on casual sneakers', color='None', material_sole='None', limited_edition='None')

In [91]:
## Se observa que si en la descripción no está la información,  el modelo devuelve None en los campos correspondientes.
print(output_3.type_shoes)
print(output_3.color)
print(output_3.material_sole)
print(output_3.limited_edition)

Slip on casual sneakers
None
None
None
